# 第 5 章 · GPU 语义：下单就走、何时会等、显存怎么借

本章回答：为什么 GPU 上的代码"跑得飞快"却测不准时间？哪些写法会让 CPU 停下来等 GPU？stream 是什么？`nvidia-smi` 看到的显存为什么比张量实际用的多？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/05-cuda-semantics.md`（讲解、动画、练习都在讲义里）。对应原文：[00-overview.md](../00-overview.md) §2.4；官方文档 CUDA semantics（〔在线〕docs.pytorch.org/docs/stable/notes/cuda.html）；AISPE ch13「使用 CUDA 流实现并发」（`ai-infra/materials/ai-systems-performance-engineering/docs/chapters/ch13.md` 第 1007–1173 行）；本章实测数字来自本机 Apple M 系列 GPU（MPS，torch 2.15.0），CUDA 上规律相同

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import time

import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 下单 vs 做完 | 不同步掐表只测到下单 | `launch_vs_done()` | 约 0.14 ms vs 10.8 ms（本机 MPS） |
| 第四步 缓存分配器 | 在用 vs 借来 | `allocator_demo()` | 128 MB → del 后 64 MB |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `gpu_device`**（来自 `ch05_cuda_semantics.py`）：下面要用到的小工具。

In [ ]:
def gpu_device():
    if torch.cuda.is_available():
        return "cuda", torch.cuda.synchronize
    if torch.backends.mps.is_available():
        return "mps", torch.mps.synchronize
    return None, None

**函数 `allocator_demo`**（来自 `ch05_cuda_semantics.py`）：返回 (两个 64 MB 张量时在用 MB, del 一个后在用 MB)。只在 MPS / CUDA 上有意义。

In [ ]:
def allocator_demo():
    """返回 (两个 64 MB 张量时在用 MB, del 一个后在用 MB)。只在 MPS / CUDA 上有意义。"""
    dev, sync = gpu_device()
    if dev is None:
        return None
    used = torch.mps.current_allocated_memory if dev == "mps" else torch.cuda.memory_allocated
    base = used()
    a = torch.randn(4096, 4096, device=dev)
    c = torch.randn(4096, 4096, device=dev)
    sync()
    two = (used() - base) / 2**20
    del c
    sync()
    one = (used() - base) / 2**20
    del a
    return two, one

**函数 `launch_vs_done`**（来自 `ch05_cuda_semantics.py`）：返回 (下单毫秒, 做完毫秒)：不同步掐表只测到下单。

In [ ]:
def launch_vs_done(n: int = 4096):
    """返回 (下单毫秒, 做完毫秒)：不同步掐表只测到下单。"""
    dev, sync = gpu_device()
    if dev is None:
        return None
    a = torch.randn(n, n, device=dev)
    for _ in range(3):
        a @ a
    sync()
    t0 = time.perf_counter()
    a @ a
    t1 = time.perf_counter()
    sync()
    t2 = time.perf_counter()
    return (t1 - t0) * 1e3, (t2 - t0) * 1e3

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 下单 vs 做完（ms）：", launch_vs_done())

In [ ]:
print("\n第四步 缓存分配器：在用（MB）", allocator_demo())

## 3. 代码里才看得到的

1. **MPS 和 CUDA 的显存函数名字不同**：`torch.mps.current_allocated_memory` 对应 `torch.cuda.memory_allocated`。
2. **数字每次略有不同**：下单约 0.1–0.2 ms、做完约 10 ms，测试只核对"做完远大于下单"。

## 4. 自己改一改

- 把 n 从 4096 改成 1024，看下单时间变不变、做完时间变多少。
- 在 `launch_vs_done` 里把 `a @ a` 换成连续 10 次，看下单时间怎么涨。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> CPU 挂单就走，GPU 排队慢做；掐表之前先同步；
> 拿数回来就要等，`item`、`print`、`if` 都是坎；
> 一条 stream 一条队，跨队用数先等待；显存仓库借了不还，在用和借来是两个数。